# Day 6: K-Nearest Neighbors & Distance Metrics

## Overview

KNN is a non-parametric algorithm that memorises training data and makes predictions based on similarity. Understand distance metrics, the curse of dimensionality, and when KNN shines vs when it struggles.

## Learning Objectives

- Understand KNN prediction mechanism
- Learn Euclidean, Manhattan, and Cosine distance
- Tune K and understand its trade-offs
- Recognise curse of dimensionality in KNN

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.decomposition import PCA

np.random.seed(42)
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. KNN Basics: K and Distance

In [ ]:
# Load Iris
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Test different K values
K_values = range(1, 31)
train_scores = []
test_scores = []

for k in K_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_values, train_scores, marker='o', label='Train', linewidth=2)
ax.plot(K_values, test_scores, marker='s', label='Test', linewidth=2)
ax.axvline(x=K_values[np.argmax(test_scores)], color='red', linestyle='--', alpha=0.5, label='Best K')
ax.set_xlabel('K (Number of Neighbors)')
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Bias-Variance Trade-off with K')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_k = K_values[np.argmax(test_scores)]
print(f"Best K: {best_k} (test accuracy: {max(test_scores):.4f})")
print(f"Rule of thumb: K ≈ sqrt(n_samples) = {int(np.sqrt(len(X_train)))}")

## 2. Distance Metrics: Euclidean, Manhattan, Cosine

In [ ]:
# Test different distance metrics
metrics = ['euclidean', 'manhattan', 'minkowski']
metric_scores = {}

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_train_scaled, y_train)
    train_score = knn.score(X_train_scaled, y_train)
    test_score = knn.score(X_test_scaled, y_test)
    metric_scores[metric] = {'train': train_score, 'test': test_score}
    print(f"Metric: {metric:15s} | Train: {train_score:.4f} | Test: {test_score:.4f}")

# Visualize
metrics_list = list(metric_scores.keys())
train_vals = [metric_scores[m]['train'] for m in metrics_list]
test_vals = [metric_scores[m]['test'] for m in metrics_list]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_list))
width = 0.35
ax.bar(x - width/2, train_vals, width, label='Train', alpha=0.8)
ax.bar(x + width/2, test_vals, width, label='Test', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_list)
ax.set_ylabel('Accuracy')
ax.set_title('KNN: Distance Metric Comparison')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 3. Curse of Dimensionality

In [ ]:
# Load high-dimensional dataset (digits: 8x8 images = 64 features)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits, y_digits, test_size=0.3, random_state=42
)

# Scale
scaler_d = StandardScaler()
X_train_d_scaled = scaler_d.fit_transform(X_train_d)
X_test_d_scaled = scaler_d.transform(X_test_d)

# Test KNN with all 64 features
knn_64 = KNeighborsClassifier(n_neighbors=5)
knn_64.fit(X_train_d_scaled, y_train_d)
score_64 = knn_64.score(X_test_d_scaled, y_test_d)

# Reduce to 2 features using PCA
pca = PCA(n_components=2)
X_train_d_pca = pca.fit_transform(X_train_d_scaled)
X_test_d_pca = pca.transform(X_test_d_scaled)

knn_2 = KNeighborsClassifier(n_neighbors=5)
knn_2.fit(X_train_d_pca, y_train_d)
score_2 = knn_2.score(X_test_d_pca, y_test_d)

print(f"KNN with 64 features: {score_64:.4f}")
print(f"KNN with 2 features (PCA): {score_2:.4f}")
print(f"\nEven though we reduced from 64 to 2 dimensions, performance didn't drop much.")
print(f"High dimensions made distances less meaningful; PCA kept only relevant variance.")

## 4. Scaling Impact on KNN

In [ ]:
# Create data where one feature has much larger range
X_unscaled = X_train.copy()
X_unscaled[:, 0] *= 100  # Scale first feature by 100x

# KNN on unscaled data
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_unscaled, y_train)
score_unscaled = knn_unscaled.score(X_unscaled, y_train)

# Prepare test set
X_test_unscaled = X_test.copy()
X_test_unscaled[:, 0] *= 100
test_unscaled = knn_unscaled.score(X_test_unscaled, y_test)

# KNN on scaled data (already done above)
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
score_scaled = knn_scaled.score(X_train_scaled, y_train)
test_scaled = knn_scaled.score(X_test_scaled, y_test)

print(f"Unscaled KNN - Train: {score_unscaled:.4f}, Test: {test_unscaled:.4f}")
print(f"Scaled KNN   - Train: {score_scaled:.4f}, Test: {test_scaled:.4f}")
print(f"\nScaling matters! Large-range features dominate distance calculations.")

## 5. KNN Complexity: Training vs Prediction Time

In [ ]:
import time

# KNN is lazy: instant training, slow prediction
knn = KNeighborsClassifier(n_neighbors=5)

# Time training
start = time.time()
knn.fit(X_train_scaled, y_train)
train_time = time.time() - start

# Time prediction
start = time.time()
_ = knn.predict(X_test_scaled)
pred_time = time.time() - start

print(f"Training time: {train_time:.6f} seconds (instant!)")
print(f"Prediction time: {pred_time:.6f} seconds (compute distances to all training points)")
print(f"\nKNN is a 'lazy learner' — stores all data, computes distances at prediction time.")
print(f"For large training sets (millions of samples), KNN prediction becomes slow.")

## Exercises

1. Train KNN with and without scaling. Measure the accuracy difference.
2. Test KNN on digits dataset. How does accuracy degrade with dimensionality?
3. Compare KNN vs Random Forest on Iris. Which is faster? Which is more accurate?

## Solutions

Key takeaways:
- Always scale KNN inputs (distance-based)
- Tune K via cross-validation
- Curse of dimensionality: use PCA or feature selection
- KNN is slow on large datasets (prediction time O(n × d))